# 1. Introduction and research objective

**Research objective.** This notebook tests whether a RIPE-style learned detector improves the repository's local Von Neumann entropy (QSED) descriptor on HPatches.  It is deliberately a **hybrid**: the learned 256-D RIPE hyper-column descriptor and its descriptor loss are not used.  Every detected point instead receives the existing configurable QSED descriptor, whose default is 24-D (`N=5`, `Y=3`).

Pipeline: **image → learned detector → grid policy sampling / NMS → QSED entropy descriptor → MNN → geometric verification**.

Paper: Künzel, Hilsmann & Eisert, *RIPE*, ICCV 2025. The detector and detector-only REINFORCE reward below follow its Sections 3.1 and 3.3; results must be generated locally, never inferred from the paper.

## 2. Existing QSED architecture explanation

The original repository remains the reference implementation for: eight-direction Sobel detection (Experiment A), diagonal intensity density matrices, Von Neumann entropy, configurable `N×N` probe neighborhoods, MNN matching, and HPatches homography metrics. The hybrid path imports `extract_all_entropy_descriptors` rather than reimplementing or changing its mathematics.

For probe patch intensity vector `v`, the repository uses `q=(v+ε)/sum(v+ε)`, `ρ=diag(q)`, and `S(ρ)=-Σ λ log₂λ`. The descriptor has exactly `N²−1` components because the center probe is excluded.

## 3. Imports and environment setup

In [1]:
# Portable setup: reuse an existing checkout/input and clone once only when necessary.
from pathlib import Path
import subprocess

REPOSITORY_URL = 'https://github.com/yaswanthvuppala/24-D-VonNeumon-Similarity-Q.git'
WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()


def _qsed_root():
    for candidate in (Path.cwd(), WORKING_DIR, Path('/kaggle/input')):
        if (candidate / 'src' / 'feature' / 'entropy_descriptor.py').is_file():
            return candidate
        if candidate.is_dir():
            matches = list(candidate.glob('**/src/feature/entropy_descriptor.py'))
            if matches:
                return matches[0].parents[2]
    return None


REPOSITORY_DIR = _qsed_root()
if REPOSITORY_DIR is None:
    REPOSITORY_DIR = WORKING_DIR / '24-D-VonNeumon-Similarity-Q'
    completed = subprocess.run(
        ['git', 'clone', '--quiet', REPOSITORY_URL, str(REPOSITORY_DIR)],
        capture_output=True, text=True,
    )
    if completed.returncode:
        raise RuntimeError(
            'Could not clone the QSED repository. Enable Internet, or add the repository as a Kaggle input.\n'
            + completed.stderr.strip()
        )
    print(f'Cloned QSED repository: {REPOSITORY_DIR}')
else:
    print(f'Using existing QSED repository: {REPOSITORY_DIR}')


Cloned QSED repository: /kaggle/working/24-D-VonNeumon-Similarity-Q


In [2]:
# If needed, run once in the notebook environment (choose the PyTorch build for your CUDA runtime):
# %pip install torch torchvision numpy scipy opencv-python matplotlib tqdm

from __future__ import annotations
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Callable, Iterable, Optional
import csv, json, math, random, sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
import torch.nn.functional as F
from torch.distributions import Bernoulli, Categorical
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19, VGG19_Weights



def find_repository_root() -> Path:
    """Locate the checked-out QSED repository in local or Kaggle notebook storage."""
    candidates = [Path.cwd(), Path('/kaggle/working'), Path('/kaggle/input')]
    for candidate in candidates:
        if (candidate / 'src' / 'feature' / 'entropy_descriptor.py').is_file():
            return candidate
        if candidate.is_dir():
            matches = list(candidate.glob('**/src/feature/entropy_descriptor.py'))
            if matches:
                return matches[0].parents[2]
    raise ModuleNotFoundError(
        'QSED source package not found. Place this notebook inside the '
        '24-D-VonNeumon-Similarity-Q repository, or add that repository as a Kaggle input. '
        'The expected file is src/feature/entropy_descriptor.py.'
    )


ROOT = find_repository_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.feature.entropy_descriptor import extract_all_entropy_descriptors, detect_default_keypoints
from src.feature.hpatches_adapter import (
    compute_corner_error, compute_homography_auc, get_hpatches_sequences,
    get_sequence_pairs, load_homography, load_hpatches_image, scale_homography,
)

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {DEVICE}; repository root: {ROOT}')


def _odd_positive(value: int, name: str) -> int:
    if not isinstance(value, int) or value <= 0 or value % 2 == 0:
        raise ValueError(f'{name} must be a positive odd integer, got {value!r}')
    return value


@dataclass
class HybridConfig:
    cell_size: int = 8
    top_k: int = 1024
    n: int = 5
    y: int = 3
    qsed_norm: str = 'l2'
    ratio_test: Optional[float] = None
    ransac_threshold: float = 1.0
    reward_rho: float = 1.0
    pretrained_encoder: bool = True
    refiner_blocks: int = 8  # RIPE reports eight DKM-inspired refiners per scale.
    learning_rate: float = 1e-3
    batch_size: int = 1
    training_steps: int = 1000

    def validate(self):
        if self.cell_size <= 0 or self.top_k <= 0 or self.ransac_threshold <= 0:
            raise ValueError('cell_size, top_k and ransac_threshold must be positive.')
        _odd_positive(self.n, 'n'); _odd_positive(self.y, 'y')
        if self.y > self.n: raise ValueError('y must not exceed n.')
        if self.qsed_norm not in {'none', 'l2', 'unit_entropy'}:
            raise ValueError('qsed_norm must be none, l2, or unit_entropy.')
        if self.ratio_test is not None and not 0 < self.ratio_test < 1:
            raise ValueError('ratio_test must lie in (0, 1).')
        return self

    @property
    def descriptor_dimension(self): return self.n * self.n - 1


CFG = HybridConfig().validate()
print(f'QSED dimension: {CFG.descriptor_dimension}')

Using cuda; repository root: /kaggle/working/24-D-VonNeumon-Similarity-Q
QSED dimension: 24


## 4. Dataset preparation

Use a CSV manifest with columns `image1,image2,label`; labels must be `+1` (same scene) or `-1` (different scenes). These weak labels do **not** claim point correspondences. HPatches may create positive pairs within a sequence and negative pairs across sequences, but it consists of mostly planar scenes, so fundamental-matrix estimation can be degenerate. The training reward therefore does not silently replace `F` with a homography. For detector training, prefer non-planar pairs with baseline/parallax (for example, MegaDepth-style or video/geotagged pairs). HPatches remains appropriate for homography evaluation.

In [3]:
class PairManifestDataset(Dataset):
    """Image-pair dataset for weak labels; no correspondence annotations are assumed."""
    def __init__(self, manifest: str | Path, image_size: Optional[int] = 560):
        self.root = Path(manifest).expanduser().resolve().parent
        with open(manifest, newline='', encoding='utf-8') as handle:
            self.rows = list(csv.DictReader(handle))
        if not self.rows or not {'image1', 'image2', 'label'} <= set(self.rows[0]):
            raise ValueError('Manifest needs image1,image2,label columns.')
        self.image_size = image_size

    def __len__(self): return len(self.rows)

    def _read(self, name: str):
        path = self.root / name
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is None: raise FileNotFoundError(path)
        original = image.copy()
        if self.image_size is not None:
            scale = self.image_size / max(image.shape)
            image = cv2.resize(image, (round(image.shape[1]*scale), round(image.shape[0]*scale)))
            # RIPE pads rather than warps: both dimensions become multiples of 8.
            pad_h = (-image.shape[0]) % 8; pad_w = (-image.shape[1]) % 8
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
        tensor = torch.from_numpy(image.astype(np.float32) / 255.0)[None]
        return tensor, image

    def __getitem__(self, index):
        row = self.rows[index]; label = int(row['label'])
        if label not in {-1, 1}: raise ValueError(f'label at row {index} must be -1 or +1.')
        a, a_gray = self._read(row['image1']); b, b_gray = self._read(row['image2'])
        return {'image1': a, 'image2': b, 'gray1': a_gray, 'gray2': b_gray, 'label': label}


def pair_collate(items):
    # QSED and OpenCV use differently sized NumPy images, so batch items stay separate.
    return items


# Example:
# train_set = PairManifestDataset('data/train_pairs.csv', image_size=560)
# train_loader = DataLoader(train_set, batch_size=CFG.batch_size, shuffle=True, collate_fn=pair_collate)

## 5. RIPE encoder–decoder implementation

In [4]:
class VGG19Encoder(nn.Module):
    """VGG-19 feature stages at strides [1, 2, 4, 8] and channels [64,128,256,512]."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = VGG19_Weights.IMAGENET1K_V1 if pretrained else None
        features = vgg19(weights=weights).features
        self.stage1 = features[:4]    # before first max pool, stride 1
        self.stage2 = features[4:9]   # pool + block 2, stride 2
        self.stage3 = features[9:18]  # pool + block 3, stride 4
        self.stage4 = features[18:27] # pool + block 4, stride 8

    def forward(self, x: Tensor):
        x1 = self.stage1(x); x2 = self.stage2(x1); x3 = self.stage3(x2); x4 = self.stage4(x3)
        return x1, x2, x3, x4


class DepthwiseRefiner(nn.Module):
    """Light DKM-inspired depthwise residual refiner used at each decode scale."""
    def __init__(self, channels: int):
        super().__init__()
        groups = min(8, channels)
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False),
            nn.Conv2d(channels, channels, 1, bias=False), nn.GroupNorm(groups, channels), nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False),
            nn.Conv2d(channels, channels, 1, bias=False), nn.GroupNorm(groups, channels),
        )

    def forward(self, x: Tensor): return F.gelu(x + self.net(x))


def refiner_stack(channels: int, blocks: int):
    if blocks < 1: raise ValueError('refiner_blocks must be at least one.')
    return nn.Sequential(*(DepthwiseRefiner(channels) for _ in range(blocks)))


class RIPEStyleDetector(nn.Module):
    """Hourglass heatmap detector only; it intentionally exposes no learned descriptor head."""
    def __init__(self, pretrained_encoder=True, refiner_blocks=8):
        super().__init__()
        self.encoder = VGG19Encoder(pretrained_encoder)
        self.p4 = nn.Conv2d(512, 512, 1)
        self.p3 = nn.Conv2d(256, 256, 1); self.f3 = nn.Conv2d(512 + 256, 256, 1); self.r3 = refiner_stack(256, refiner_blocks)
        self.p2 = nn.Conv2d(128, 128, 1); self.f2 = nn.Conv2d(256 + 128, 128, 1); self.r2 = refiner_stack(128, refiner_blocks)
        self.p1 = nn.Conv2d(64, 64, 1); self.f1 = nn.Conv2d(128 + 64, 64, 1); self.r1 = refiner_stack(64, refiner_blocks)
        self.head = nn.Conv2d(64, 1, 1)

    def forward(self, image: Tensor) -> Tensor:
        if image.ndim != 4 or image.shape[1] not in (1, 3): raise ValueError('Expected B×1/3×H×W input.')
        if image.shape[1] == 1: image = image.repeat(1, 3, 1, 1)
        x1, x2, x3, x4 = self.encoder(image)
        h3 = self.r3(self.f3(torch.cat([F.interpolate(self.p4(x4), size=x3.shape[-2:], mode='bilinear', align_corners=False), self.p3(x3)], 1)))
        h2 = self.r2(self.f2(torch.cat([F.interpolate(h3, size=x2.shape[-2:], mode='bilinear', align_corners=False), self.p2(x2)], 1)))
        h1 = self.r1(self.f1(torch.cat([F.interpolate(h2, size=x1.shape[-2:], mode='bilinear', align_corners=False), self.p1(x1)], 1)))
        return F.interpolate(self.head(h1), size=image.shape[-2:], mode='bilinear', align_corners=False).squeeze(1)


def build_detector(cfg=CFG):
    return RIPEStyleDetector(cfg.pretrained_encoder, cfg.refiner_blocks).to(DEVICE)

## 6. Heatmap generation

`H = Decoder(Encoder(I))` returns location logits at input resolution. The cell policy below performs the discrete action sampling used by REINFORCE.

## 7. Probabilistic keypoint sampling

Each `m×m` heatmap cell defines a categorical distribution. One location is sampled per cell during training, then accepted with probability `sigmoid(lᵢ)`. The recorded log-policy probability includes both the categorical action and the Bernoulli accept/reject action.

In [5]:
@dataclass
class SampledPolicy:
    xy: list[Tensor]             # accepted x,y coordinates per image
    log_prob: list[Tensor]       # log P(location and acceptance)
    confidence: list[Tensor]     # sigmoid(logit) * categorical probability
    accepted: list[Tensor]


def sample_grid_policy(logits: Tensor, cell_size: int = 8) -> SampledPolicy:
    """Training policy: one categorical draw and one Bernoulli acceptance draw per cell."""
    if logits.ndim != 3: raise ValueError('logits must have shape B×H×W.')
    b, h, w = logits.shape; m = cell_size
    ph, pw = (-h) % m, (-w) % m
    padded = F.pad(logits, (0, pw, 0, ph), value=-torch.inf)
    gh, gw = padded.shape[-2] // m, padded.shape[-1] // m
    cells = padded.reshape(b, gh, m, gw, m).permute(0, 1, 3, 2, 4).reshape(b, gh*gw, m*m)
    categorical = Categorical(logits=cells)
    choice = categorical.sample()
    choice_logit = cells.gather(-1, choice[..., None]).squeeze(-1)
    accept_dist = Bernoulli(logits=choice_logit)
    accepted = accept_dist.sample().bool()
    log_policy = categorical.log_prob(choice) + accept_dist.log_prob(accepted.float())
    category_prob = categorical.probs.gather(-1, choice[..., None]).squeeze(-1)
    confidence = category_prob * torch.sigmoid(choice_logit)
    cy, cx = torch.div(choice, m, rounding_mode='floor'), choice % m
    cell_y = torch.arange(gh, device=logits.device)[None, :, None].expand(b, gh, gw).reshape(b, -1)
    cell_x = torch.arange(gw, device=logits.device)[None, None, :].expand(b, gh, gw).reshape(b, -1)
    ys, xs = cell_y*m + cy, cell_x*m + cx
    result_xy, result_lp, result_conf, result_accept = [], [], [], []
    for i in range(b):
        valid = accepted[i] & (xs[i] < w) & (ys[i] < h)
        result_xy.append(torch.stack((xs[i, valid], ys[i, valid]), -1))
        result_lp.append(log_policy[i, valid]); result_conf.append(confidence[i, valid]); result_accept.append(valid)
    return SampledPolicy(result_xy, result_lp, result_conf, result_accept)


def infer_keypoints(logits: Tensor, top_k: int = 1024, nms_size: int = 3):
    """RIPE-style inference: p=softmax(cell logits)*sigmoid(logit), 3×3 NMS, then top-K."""
    if logits.ndim != 2: raise ValueError('Expected one H×W heatmap.')
    h, w = logits.shape
    probability = torch.softmax(logits.flatten(), 0).reshape_as(logits) * torch.sigmoid(logits)
    pooled = F.max_pool2d(probability[None, None], nms_size, stride=1, padding=nms_size//2)[0, 0]
    keep = (probability == pooled) & torch.isfinite(probability)
    score = probability.masked_fill(~keep, -torch.inf).flatten()
    count = min(top_k, int(torch.isfinite(score).sum()))
    values, indices = torch.topk(score, count)
    xy = torch.stack((indices % w, torch.div(indices, w, rounding_mode='floor')), 1)
    return xy.cpu().numpy().astype(np.int32), values.detach().cpu().numpy(), probability


# Acceptance mechanism used by both sampling and inference: p_i = sigmoid(l_i) * p_hat_i.

## 8. Acceptance mechanism

The acceptance term rejects uninformative regions even though a categorical location is drawn from every training cell. At inference, the same combined probability is scored globally, followed by 3×3 NMS and top-`K` selection.

## 9. QSED density matrix implementation

## 10. Von Neumann entropy extraction

The repository computes `S(ρ) = -Tr(ρ log₂ ρ)` from the eigenvalues of each valid density matrix. This notebook calls that existing implementation through the descriptor API.

## 11. QSED descriptor generation

The `qsed_describe` adapter below passes every learned-detector coordinate to the unchanged QSED implementation. It supports arbitrary odd `N` and `Y` values (`N²−1` dimensions) and preserves the repository’s reflection-padding boundary strategy.

In [6]:
def qsed_describe(gray: np.ndarray, xy: np.ndarray, cfg=CFG) -> np.ndarray:
    """Call the repository's unchanged configurable density-matrix/entropy descriptor."""
    gray = np.asarray(gray)
    if gray.ndim != 2 or xy.ndim != 2 or xy.shape[1] != 2: raise ValueError('Need grayscale image and K×2 xy coordinates.')
    if len(xy) == 0: return np.empty((0, cfg.descriptor_dimension), dtype=np.float32)
    return extract_all_entropy_descriptors(gray, [tuple(map(int, p)) for p in xy],
                                           norm=cfg.qsed_norm, n=cfg.n, y=cfg.y).astype(np.float32)


def extract_hybrid_features(model: nn.Module, gray: np.ndarray, cfg=CFG):
    """Inference-only learned detector plus handcrafted QSED descriptor; returns no RIPE descriptor."""
    model.eval()
    image = torch.from_numpy(np.asarray(gray, dtype=np.float32) / 255.0)[None, None].to(DEVICE)
    with torch.no_grad(): logits = model(image)[0]
    xy, confidence, probability = infer_keypoints(logits, cfg.top_k)
    descriptors = qsed_describe(gray, xy, cfg)
    return {'xy': xy, 'scores': confidence, 'descriptors': descriptors,
            'heatmap_logits': logits.cpu().numpy(), 'probability': probability.cpu().numpy()}


def save_features(path: str | Path, features: dict):
    np.savez_compressed(path, xy=features['xy'], scores=features['scores'], descriptors=features['descriptors'])


# QSED padding is performed by extract_all_entropy_descriptors, preserving valid border descriptors.

## 12. MNN matching

In [7]:
def mnn_match(desc_a: np.ndarray, desc_b: np.ndarray, ratio: Optional[float] = None):
    """Euclidean mutual-nearest-neighbour matches; ratio is optional and disabled by default."""
    if len(desc_a) == 0 or len(desc_b) == 0: return np.empty((0, 2), np.int32), np.empty(0, np.float32)
    distance = np.linalg.norm(desc_a[:, None, :] - desc_b[None, :, :], axis=2)
    a_to_b, b_to_a = distance.argmin(1), distance.argmin(0)
    ia = np.arange(len(desc_a)); keep = b_to_a[a_to_b] == ia
    if ratio is not None and len(desc_b) >= 2:
        two = np.partition(distance, 1, axis=1)[:, :2]
        keep &= two[:, 0] / np.maximum(two[:, 1], 1e-12) < ratio
    pairs = np.column_stack((ia[keep], a_to_b[keep])).astype(np.int32)
    return pairs, distance[pairs[:, 0], pairs[:, 1]].astype(np.float32) if len(pairs) else np.empty(0, np.float32)


def fundamental_ransac(xy_a: np.ndarray, xy_b: np.ndarray, matches: np.ndarray, threshold=1.0):
    """OpenCV 8-point RANSAC F estimation; returns no fake geometry below eight matches."""
    if len(matches) < 8: return None, np.zeros(len(matches), dtype=bool)
    points_a, points_b = xy_a[matches[:, 0]].astype(np.float32), xy_b[matches[:, 1]].astype(np.float32)
    F_matrix, mask = cv2.findFundamentalMat(points_a, points_b, cv2.FM_RANSAC, threshold, 0.999)
    if F_matrix is None or mask is None: return None, np.zeros(len(matches), dtype=bool)
    return F_matrix, mask.ravel().astype(bool)

## 13. Fundamental matrix estimation and RANSAC

For the reinforcement reward we preserve RIPE's fundamental-matrix check. HPatches evaluation separately estimates its provided planar homography, because that is the benchmark protocol. The two are not interchangeable.

## 14. REINFORCE reward calculation

For each geometrically valid MNN pair `(p,q)`, the hybrid uses `r_pq = sign(λ)ρ`; all other entries are zero. Since QSED is handcrafted and non-differentiable, this notebook implements the detector-only REINFORCE part of RIPE Eq. 6 and intentionally omits RIPE's learned-descriptor auxiliary loss (Eq. 8).

In [8]:
def reinforce_detector_loss(policy_a: SampledPolicy, policy_b: SampledPolicy, gray_a: np.ndarray, gray_b: np.ndarray, label: int, cfg=CFG):
    """Non-differentiable QSED/MNN/F reward, differentiable log-policy loss for detector parameters."""
    if label not in {-1, 1}: raise ValueError('label must be -1 or +1.')
    xy_a = policy_a.xy[0].detach().cpu().numpy().astype(np.int32)
    xy_b = policy_b.xy[0].detach().cpu().numpy().astype(np.int32)
    desc_a, desc_b = qsed_describe(gray_a, xy_a, cfg), qsed_describe(gray_b, xy_b, cfg)
    matches, _ = mnn_match(desc_a, desc_b, cfg.ratio_test)
    _, inliers = fundamental_ransac(xy_a, xy_b, matches, cfg.ransac_threshold)
    if not np.any(inliers):
        # A zero reward is correct: no policy-gradient contribution for this trajectory.
        return policy_a.log_prob[0].sum() * 0.0, {'matches': matches, 'inliers': inliers, 'reward': 0.0}
    valid = matches[inliers]
    reward = float(np.sign(label) * cfg.reward_rho)
    log_joint = policy_a.log_prob[0][valid[:, 0]] + policy_b.log_prob[0][valid[:, 1]]
    # Minimise -E[R log pi]; averaging controls gradient scale across pairs.
    loss = -(reward * log_joint).mean()
    return loss, {'matches': matches, 'inliers': inliers, 'reward': reward, 'num_inliers': int(inliers.sum())}


def training_step(model, optimizer, item, cfg=CFG):
    model.train(); optimizer.zero_grad(set_to_none=True)
    a, b = item['image1'][None].to(DEVICE), item['image2'][None].to(DEVICE)
    policy_a = sample_grid_policy(model(a), cfg.cell_size)
    policy_b = sample_grid_policy(model(b), cfg.cell_size)
    loss, info = reinforce_detector_loss(policy_a, policy_b, item['gray1'], item['gray2'], int(item['label']), cfg)
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
    return float(loss.detach()), info


def save_checkpoint(path, model, optimizer, step, cfg=CFG):
    torch.save({'step': step, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'config': asdict(cfg)}, path)


def train_detector(model, loader, cfg=CFG, checkpoint_path='checkpoints/qsed_ripe_detector.pt'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)
    checkpoint_path = Path(checkpoint_path); checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    history, step = [], 0
    while step < cfg.training_steps:
        for batch in loader:
            for item in batch:
                loss, info = training_step(model, optimizer, item, cfg); history.append({'step': step, 'loss': loss, **info}); step += 1
                if step % 100 == 0: save_checkpoint(checkpoint_path, model, optimizer, step, cfg)
                if step >= cfg.training_steps: break
            if step >= cfg.training_steps: break
    save_checkpoint(checkpoint_path, model, optimizer, step, cfg)
    return history


# GPU planning: the paper used 560px inputs, batch 6, gradient accumulation 4, and an A100 for a long run.
# Start with one 560px pair, refiner_blocks=1, and 10 steps; full VGG-19 training needs substantially more VRAM/time.

## 15. Detector training

In [9]:
# This executes a real shape/policy/QSED check without downloading ImageNet weights or claiming training success.
SMOKE_TEST = True
if SMOKE_TEST:
    smoke_cfg = HybridConfig(pretrained_encoder=False, refiner_blocks=1, top_k=32).validate()
    smoke_model = build_detector(smoke_cfg).eval()
    smoke_image = (np.random.default_rng(SEED).random((64, 64)) * 255).astype(np.uint8)
    with torch.no_grad(): smoke_logits = smoke_model(torch.from_numpy(smoke_image/255.0).float()[None, None].to(DEVICE))
    smoke_xy, smoke_scores, _ = infer_keypoints(smoke_logits[0], smoke_cfg.top_k)
    smoke_desc = qsed_describe(smoke_image, smoke_xy, smoke_cfg)
    assert smoke_logits.shape == (1, 64, 64) and smoke_desc.shape == (len(smoke_xy), 24)
    print(f'Smoke test passed: {len(smoke_xy)} points, descriptors {smoke_desc.shape}.')

Smoke test passed: 32 points, descriptors (32, 24).


### Training invocation

Do not run this until the pair manifest has been inspected and a dataset suitable for epipolar geometry is selected. Binary labels alone are the supervision; the RANSAC result creates a reward, not ground-truth correspondences.

In [10]:
# Example (intentionally disabled until you set a real manifest):
# model = build_detector(CFG)
# loader = DataLoader(PairManifestDataset('data/train_pairs.csv'), batch_size=CFG.batch_size, shuffle=True, collate_fn=pair_collate)
# history = train_detector(model, loader, CFG)
# No evaluation/training claim is made until these cells complete with a real dataset.

## 16. Keypoint inference

In [11]:
def plot_detector_stages(gray, features, sampled_xy=None):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Input')
    axes[1].imshow(features['heatmap_logits'], cmap='magma'); axes[1].set_title('RIPE-style heatmap logits')
    axes[2].imshow(features['probability'], cmap='magma'); axes[2].set_title('Combined selection probability')
    axes[3].imshow(gray, cmap='gray'); axes[3].scatter(features['xy'][:,0], features['xy'][:,1], s=8, c='lime'); axes[3].set_title('Final top-K after 3×3 NMS')
    if sampled_xy is not None and len(sampled_xy): axes[3].scatter(sampled_xy[:,0], sampled_xy[:,1], s=8, c='orange', label='sampled'); axes[3].legend()
    for ax in axes: ax.axis('off')
    plt.tight_layout(); return fig


def plot_matches(image_a, image_b, xy_a, xy_b, matches, inliers=None, max_lines=100):
    canvas = np.hstack([image_a, image_b]); width = image_a.shape[1]
    fig, ax = plt.subplots(figsize=(16, 7)); ax.imshow(canvas, cmap='gray')
    for index, (ia, ib) in enumerate(matches[:max_lines]):
        good = inliers is None or bool(inliers[index]); color = 'lime' if good else 'red'
        a, b = xy_a[ia], xy_b[ib]; ax.plot([a[0], b[0]+width], [a[1], b[1]], color=color, linewidth=.7)
    ax.set_title('MNN matches: green=geometrically verified, red=other'); ax.axis('off'); return fig


def plot_original_qsed(gray, max_keypoints=1024):
    points = detect_default_keypoints(gray, max_keypoints=max_keypoints)
    xy = np.array([(p.x, p.y) for p in points], dtype=np.int32)
    fig, ax = plt.subplots(figsize=(7, 5)); ax.imshow(gray, cmap='gray');
    if len(xy): ax.scatter(xy[:, 0], xy[:, 1], s=7, c='cyan')
    ax.set_title('Original Sobel/directional-variation QSED keypoints'); ax.axis('off'); return fig, points

## 17. HPatches evaluation

The evaluation uses the benchmark’s homographies for repeatability and homography estimation. It reports threshold accuracy (`error ≤ T`) separately from normalized integrated AUC (`1/T ∫₀ᵀ recall(t) dt`); they are not the same metric.

In [12]:
def repeatability(xy_a, xy_b, H, threshold=3.0):
    if len(xy_a) == 0 or len(xy_b) == 0: return 0.0
    homogeneous = np.c_[xy_a, np.ones(len(xy_a))] @ H.T
    projected = homogeneous[:, :2] / np.maximum(homogeneous[:, 2:3], 1e-12)
    distance = np.linalg.norm(projected[:, None] - xy_b[None], axis=2)
    return float((distance.min(axis=1) <= threshold).sum() / max(1, min(len(xy_a), len(xy_b))))


def homography_ransac(xy_a, xy_b, matches, threshold=3.0):
    if len(matches) < 4: return None, np.zeros(len(matches), bool)
    H, mask = cv2.findHomography(xy_a[matches[:,0]], xy_b[matches[:,1]], cv2.RANSAC, threshold)
    return H, np.zeros(len(matches), bool) if mask is None else mask.ravel().astype(bool)


def summarize_hpatches(errors):
    errors = np.asarray(errors, dtype=float)
    result = {}
    for threshold in (1, 3, 5):
        result[f'accuracy@{threshold}px'] = float(np.mean(errors <= threshold) * 100)
        # Repository helper implements the HPatches cumulative-recall trapezoidal AUC protocol.
        result[f'AUC@{threshold}px'] = float(compute_homography_auc(errors, [threshold])[f'AUC@{threshold}px'])
    result['mean_reprojection_error'] = float(np.mean(errors[np.isfinite(errors)])) if np.isfinite(errors).any() else float('inf')
    return result


def evaluate_hpatches_hybrid(model, dataset_dir, cfg=CFG, max_side=480):
    records, corner_errors = [], []
    for sequence in get_hpatches_sequences(dataset_dir):
        reference, sr = load_hpatches_image(str(Path(sequence['path']) / '1.ppm'), max_side=max_side)
        ref = extract_hybrid_features(model, reference, cfg)
        for pair in get_sequence_pairs(sequence['path']):
            target, st = load_hpatches_image(pair['tgt_path'], max_side=max_side)
            tgt = extract_hybrid_features(model, target, cfg)
            matches, _ = mnn_match(ref['descriptors'], tgt['descriptors'], cfg.ratio_test)
            H_est, inliers = homography_ransac(ref['xy'], tgt['xy'], matches, cfg.ransac_threshold)
            H_gt = scale_homography(load_homography(pair['H_path']), sr, st)
            error = compute_corner_error(H_gt, H_est, reference.shape) if H_est is not None else float('inf')
            corner_errors.append(error)
            records.append({'sequence': sequence['name'], 'pair': pair['pair_name'], 'detected_1': len(ref['xy']), 'detected_2': len(tgt['xy']),
                            'descriptors_1': len(ref['descriptors']), 'descriptor_dimension': cfg.descriptor_dimension,
                            'tentative_matches': len(matches), 'verified_matches': int(inliers.sum()),
                            'repeatability': repeatability(ref['xy'], tgt['xy'], H_gt), 'corner_reprojection_error': error})
    return records, summarize_hpatches(corner_errors)


# Example after training: records, metrics = evaluate_hpatches_hybrid(model, 'data/hpatches-sequences-release')
# print(metrics)

## 18. Original QSED vs Hybrid QSED–RIPE comparison

Experiment A uses the repository's Sobel/directional-variation detector. Experiment B replaces only that detector with the learned heatmap model while retaining the same QSED `N=5, Y=3` descriptor and matching configuration.

In [13]:
def extract_original_qsed_features(gray, cfg=CFG):
    points = detect_default_keypoints(gray, max_keypoints=cfg.top_k)
    xy = np.asarray([(point.x, point.y) for point in points], dtype=np.int32).reshape(-1, 2)
    return {'xy': xy, 'scores': np.asarray([point.keypoint_score for point in points]), 'descriptors': qsed_describe(gray, xy, cfg)}


def feature_summary(features):
    return {'detected_keypoints': len(features['xy']), 'descriptors': len(features['descriptors']),
            'descriptor_dimension': int(features['descriptors'].shape[1]) if len(features['descriptors']) else CFG.descriptor_dimension}

# Use extract_original_qsed_features(...) for A and extract_hybrid_features(model, ...) for B under identical N, Y, K, MNN, and RANSAC settings.

## 19. Ablation studies

Experiment C changes only the QSED probe neighborhood over the full grid `N ∈ {5, 7, 9, 11, 13}` and `Y ∈ {3, 5}`. Descriptor dimensions are `N²−1`: 24-D, 48-D, 80-D, 120-D, and 168-D respectively. No benchmark numbers are filled in by this notebook.

In [14]:
ABLATION_N_VALUES = (5, 7, 9, 11, 13)
ABLATION_Y_VALUES = (3, 5)


def descriptor_ablation_configs(base=CFG, n_values=ABLATION_N_VALUES, y_values=ABLATION_Y_VALUES):
    """Return the full N × Y descriptor-ablation grid, with dimension N² − 1."""
    return [
        HybridConfig(**{**asdict(base), 'n': n, 'y': y}).validate()
        for n in n_values for y in y_values
    ]


def descriptor_ablation_plan(base=CFG):
    """Display the ten requested configurations before starting an evaluation run."""
    return [
        {'N': cfg.n, 'Y': cfg.y, 'dimension': cfg.descriptor_dimension}
        for cfg in descriptor_ablation_configs(base)
    ]


def run_descriptor_ablation(model, dataset_dir, base=CFG, max_side=480):
    """Evaluate one trained detector over all requested QSED N/Y settings."""
    rows = []
    for ablation_cfg in descriptor_ablation_configs(base):
        records, metrics = evaluate_hpatches_hybrid(model, dataset_dir, ablation_cfg, max_side)
        rows.append({'N': ablation_cfg.n, 'Y': ablation_cfg.y,
                     'dimension': ablation_cfg.descriptor_dimension, 'pairs': len(records), **metrics})
    return rows


def plot_dimension_comparison(result_rows):
    """result_rows should contain measured fields: dimension and AUC@5px (or another chosen metric)."""
    dims = [row['dimension'] for row in result_rows]; values = [row['AUC@5px'] for row in result_rows]
    fig, ax = plt.subplots(figsize=(6, 4)); ax.bar([str(d) for d in dims], values); ax.set(xlabel='QSED descriptor dimension', ylabel='Measured AUC@5px (%)', title='Descriptor dimensionality ablation')
    return fig


# The ten requested configurations (run under identical detector/matching/RANSAC settings):
# print(descriptor_ablation_plan())
# results = run_descriptor_ablation(model, 'data/hpatches-sequences-release')
# N=5, 7, 9, 11, 13; Y=3, 5; descriptor dimensions are 24, 48, 80, 120, 168 respectively.
# Suggested result table columns:
# method | detector | N | Y | dimension | detected_keypoints | descriptors | tentative_matches | verified_matches | repeatability | mean_reprojection_error | accuracy@1/3/5px | AUC@1/3/5px

## 20. Conclusion and research observations

This executable design preserves QSED’s handcrafted entropy formulation while replacing Sobel scoring only in the hybrid experiment. A trained detector is not provided and no performance conclusion is encoded: use the smoke test to validate the pipeline, train only on non-planar weakly-labeled pairs suitable for an epipolar reward, then evaluate A/B/C with the exact same HPatches settings. The key research question remains empirical: does learned point selection make the 24-D QSED descriptor more geometrically matchable?